In [1]:
import xarray as xr
import intake
import healpix as hp
import numpy as np
cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")["online"]
# List a specific model simulations
[key for key in cat if key.startswith('icon')]

['icon_d3hp003', 'icon_d3hp003aug', 'icon_d3hp003feb', 'icon_ngc4008']

In [2]:
def attach_crs(dataset:xr.Dataset, zoom:int, drop_vars_logical:bool)->xr.Dataset:
    """You will need to re-assign the return value to the dataset!"""
    # Createa a coordinate-references system (CRS)
    crs = xr.DataArray(
        name="crs",
        attrs={
            "grid_mapping_name": "healpix",
            "healpix_nside": 2**zoom,
            "healpix_order": "nest",
        },
    )

    if drop_vars_logical:
        # Drop existing lon/lat coordinates (if existing)
        dataset = dataset.drop_vars(["lat", "lon"], errors="ignore")

    # Fix dimension name
    if "value" in dataset.dims:
        dataset = dataset.rename(value="cell")
    
    return dataset.assign_coords(crs=crs)

In [3]:
zoom = 7 # MOST MODELS
ds = cat['icon_d3hp003'](zoom=zoom,  time='PT3H').to_dask().pipe(attach_crs , zoom=zoom, drop_vars_logical=True)
# Remove all boolean attributes from the dataset before writing
for v in ds.data_vars:
    for k, val in list(ds[v].attrs.items()):
        if isinstance(val, bool):
            ds[v].attrs[k] = int(val)
            
lon = np.arange(0,360,0.5)
lat = np.arange(-15,15,0.5)
pix = xr.DataArray(
    hp.ang2pix(ds.crs.healpix_nside, *np.meshgrid(lon, lat), nest=True, lonlat=True),
    coords=(("lat", lat), ("lon", lon)))
rlut_lon_lat = ds.rlut.isel(cell=pix,time=slice(None, None, 2)) #FOR ICON 3-hourly
rlut_lon_lat.to_netcdf("/gws/nopw/j04/hrcm/mmuetz/wavenumber_frequency/icon_d3hp003_rlut_latlon.nc")

/home/users/mmuetz/miniforge3/envs/hackathon_base_env/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/home/users/mmuetz/miniforge3/envs/hackathon_base_env/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [4]:
debug

ERROR:root:No traceback has been produced, nothing to debug.


In [5]:
ds

<xarray.Dataset> Size: 123GB
Dimensions:        (time: 3400, cell: 196608, soil_level: 5)
Coordinates:
  * soil_level     (soil_level) int64 40B 0 0 0 2 6
  * time           (time) datetime64[ns] 27kB 2020-01-01T03:00:00 ... 2021-03-01
    crs            float64 8B nan
Dimensions without coordinates: cell
Data variables: (12/45)
    clivi          (time, cell) float32 3GB ...
    clt            (time, cell) float32 3GB ...
    clwvi          (time, cell) float32 3GB ...
    hflsd          (time, cell) float32 3GB ...
    hfssd          (time, cell) float32 3GB ...
    huss           (time, cell) float32 3GB ...
    ...             ...
    tend_ekhdynvi  (time, cell) float32 3GB ...
    tend_ekhtmxvi  (time, cell) float32 3GB ...
    tend_ekvdynvi  (time, cell) float32 3GB ...
    ts             (time, cell) float32 3GB ...
    uas            (time, cell) float32 3GB ...
    vas            (time, cell) float32 3GB ...

In [6]:
ds = cat['nicam_gl11'](zoom=zoom, time='PT1H').to_dask().pipe(attach_crs , zoom=zoom)

/home/users/mmuetz/miniforge3/envs/hackathon_base_env/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


TypeError: attach_crs() missing 1 required positional argument: 'drop_vars_logical'

In [ ]:
ds = cat['ifs_tco3999-ng5_rcbmf_cf'](zoom=zoom).to_dask().pipe(attach_crs , zoom=zoom, drop_vars_logical=False)

In [ ]:
ds.time

In [ ]:
cat['nicam_gl11']

In [ ]:
ds = cat['icon_d3hp003']

In [ ]:
ds

In [ ]:
ds = cat['icon_d3hp003'](zoom=zoom).to_dask().pipe(attach_crs , zoom=zoom, drop_vars_logical=False)

In [ ]:
ds